# 《实用Python编程》教学代码
## 第8章 大模型二次开发

## 示例代码8.1 通过openai 调用云端大模型
演示如何远程调用千问大模型。调用方式可能随模型迭代发生改变，最新版见[官方开发文档](https://help.aliyun.com/zh/model-studio/qwen-api-reference)。

In [130]:
from openai import OpenAI

class LLMClient():
    def __init__(self, params):
        self.client = OpenAI(api_key=params['apiKey'], base_url=params['openAiCompatible'])    
        
    def _process (self, prompt, model="qwen3-max"):
        messages = [{"role": "system", "content": "You are a helpful assistant."}, {"role": "user", "content":prompt}]
        completion = self.client.chat.completions.create(model=model, messages=messages)
        resp = completion.to_dict()['choices'][0]['message']['content']
        return resp

    def answer(self, prompt):
        return self._process(prompt)

### LLMClient的测试代码

In [ ]:
import pandas as pd

project_id = 'Your-Project-ID'
key_file = f'默认业务空间-apiKey-{project_id}.csv' # 从
# 将CSV文件中的第1列（索引为0的列）设为DataFrame的行索引
df = pd.read_csv(key_file, header=None, index_col=0) 
# 选择列名为 1 的列（即CSV文件中的第2列数据），并转换为字典，格式为 {索引: 值}
params = df[1].to_dict() 
print(params['openAiCompatible'], params['apiKey'])

llm = LLMClient(params)
resp = llm.answer("最小的鸟类是什么？")
print(resp)

世界上最小的鸟类是**蜂鸟**中的**吸蜜蜂鸟**（学名：Mellisuga helenae），也被称为**古巴蜜蜂鸟**。

以下是关于吸蜜蜂鸟的一些关键信息：

- **体长**：约5—6厘米（包括喙和尾羽）。
- **体重**：仅约1.6—2克，比一枚硬币还轻。
- **分布**：仅分布于古巴及其邻近的小岛。
- **特征**：雄鸟羽毛色彩鲜艳，具有金属光泽的绿色和红色；雌鸟则颜色较暗淡。
- **习性**：以花蜜为食，飞行能力极强，可以悬停、倒飞，翅膀每秒可拍动80次以上。

由于其体型极小、新陈代谢极快，吸蜜蜂鸟每天需要摄取相当于自身体重一半以上的花蜜来维持能量。

因此，**吸蜜蜂鸟被公认为是世界上体型最小的鸟类**。

## 使用huggingface_hub 下载大模型（Qwen3-0.6B）到本地（./local_llms/Qwen3-0.6B）

In [8]:
from huggingface_hub import snapshot_download

snapshot_download(repo_id="Qwen/Qwen3-0.6B", local_dir="./local_llms/Qwen3-0.6B", endpoint='https://hf-mirror.com')

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

'/Users/xirong/workspace/py2026fall/local_llms/Qwen3-0.6B'

## 示例代码8.2 通过transformers调用本地大模型

In [73]:
from transformers import AutoModelForCausalLM, AutoTokenizer  
        
class LocalLLMClient(LLMClient):
    def __init__(self, model_path):
        self.tokenizer = AutoTokenizer.from_pretrained(model_path)
        self.model = AutoModelForCausalLM.from_pretrained(model_path, dtype="auto", device_map="auto")

    def _process(self, prompt):
        messages = [{"role": "user", "content": prompt}]
        text = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
        model_inputs = self.tokenizer([text], return_tensors="pt").to(self.model.device)
        generated_ids = self.model.generate(**model_inputs, max_new_tokens=10000) #, do_sample=False)
        output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 
        resp = self.tokenizer.decode(output_ids, skip_special_tokens=True)
        return resp
        
# 测试代码
model_path = 'local_llms/Qwen3-0.6B'
llm = LocalLLMClient(model_path)
resp = llm.answer("最小的鸟类是什么？") 
print(resp)

最小的鸟类是**鹦鹉**。


In [45]:
import torch
import random
import numpy as np

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

# 在初始化模型和调用前，先锁死种子
set_seed()

model_path = 'local_llms/Qwen3-0.6B'
llm = LocalLLMClient(model_path)
resp = llm.answer("最小的鸟类是什么？") 
print(resp)

最小的鸟类是**麻雀**（*Mimus*）。


## 示例代码8.3 构建关于上市公司股价的迷你知识库

### 方案1. 基于akshare库获取并构建股票数据库

In [ ]:
import akshare as ak   # 通过pip install akshare安装

years = [2024, 2025]  # 要获取数据的年份
stock_list = [('601398','工商银行'), ('600519','贵州茅台'), ('300750','宁德时代')] # 要获取数据的(股票代码、公司名称）二元组列表
stock_data_path = 'stock_data_akshare.txt' # 存储地址
with open(stock_data_path, 'w') as fw:
    for stock_id, company in stock_list:
        for year in years:
            df = ak.stock_zh_a_hist(symbol=stock_id, period='monthly',
                start_date=f'{year}0101', end_date=f'{year}1231', adjust='qfq')
            texts = [f"日期{row['日期']}，收盘价{row['收盘']}元。" for _,row in df.iterrows()]
            texts.insert(0, f'{year}年{company}（{stock_id}）月度股票行情数据。')
            fw.write(''.join(texts)  + '\n')

### 方案2. 基于腾讯财经接口获取并构建股票数据库

#### 自定义类 TFShare 

In [72]:
import pandas as pd
import requests

RENAME_PERIOD = {'daily': 'day', 'weekly': 'week', 'monthly': 'month'}
        
class TFShare():
    @staticmethod
    def __get_market_code(stock_id):
        """由6位股票代码推断股票所在交易所：sh（上交所）/sz（深交所）/bj（北交所）"""
        if stock_id.startswith(('60', '68', '90', '11', '13')):
            return 'sh'
        if stock_id.startswith(('00', '30', '20', '12')):
            return 'sz'
        return 'bj'
    
    """腾讯财经行情接口，stock_zh_a_hist方法与akshare的同名函数用法一致。"""
    @staticmethod
    def stock_zh_a_hist(symbol, period='daily', start_date='19700101', end_date='20500101', adjust='qfq'):
        assert(adjust in ['', 'qfq', 'hfq'])
        code = TFShare.__get_market_code(symbol) + symbol
        period = RENAME_PERIOD[period]
        begin, finish = [f'{d[:4]}-{d[4:6]}-{d[6:8]}' for d in (start_date, end_date)] # 腾讯接口的日期格式
        url = 'https://web.ifzq.gtimg.cn/appstock/app/fqkline/get'
        param = f'{code},{period},{begin},{finish},1000,{adjust}'
        klines = []
        try:
            response = requests.get(url, params={'param': param}, timeout=10)
            response.raise_for_status()   # 如果返回状态码不是200，则抛出异常
            result = response.json()
            klines = result['data'][code].get(f'{adjust}{period}', []) 
        except Exception as e:
            print(f"请求失败，错误信息: {e}")
        columns = ['日期', '开盘', '收盘', '最高', '最低', '成交量']
        df = pd.DataFrame([kline[:6] for kline in klines], columns=columns)
        df['日期'] = pd.to_datetime(df['日期'], errors='coerce').dt.date
        for col in columns[1:]:
            df[col] = pd.to_numeric(df[col], errors='coerce')
        return df

# 测试代码
year = 2025
stock_id = '601398'
df = TFShare.stock_zh_a_hist(symbol=stock_id, period='monthly',start_date=f'{year}0101', end_date=f'{year}1231')
df

,日期,开盘,收盘,最高,最低,成交量
0,2025-01-27,6.292,6.345,6.375,5.925,64319547.0
1,2025-02-28,6.345,6.395,6.665,6.135,63131259.0
2,2025-03-31,6.395,6.415,6.465,6.225,67024784.0
3,2025-04-30,6.395,6.535,6.865,6.125,84388556.0
4,2025-05-30,6.535,6.595,6.825,6.455,53944359.0
5,2025-06-30,6.575,7.115,7.285,6.545,59525517.0
6,2025-07-31,7.115,7.250,7.885,7.090,83537190.0
7,2025-08-29,7.250,7.120,7.590,7.010,81947790.0
8,2025-09-30,7.120,6.990,7.290,6.810,92550903.0
9,2025-10-31,6.940,7.470,7.710,6.890,66212344.0


#### 利用TFShare获取并构建股票数据库

In [60]:
years = [2024, 2025]  # 要获取数据的年份
stock_list = [('601398','工商银行'), ('600519','贵州茅台'), ('300750','宁德时代')] # 要获取数据的(股票代码、公司名称）二元组列表
stock_data_path = 'stock_data_tfshare.txt' # 存储地址
with open(stock_data_path, 'w') as fw:
    for stock_id, company in stock_list:
        for year in years:
            df = TFShare.stock_zh_a_hist(symbol=stock_id, period='monthly',
                start_date=f'{year}0101', end_date=f'{year}1231', adjust='qfq')
            texts = [f"日期{row['日期']}，收盘价{row['收盘']}元。" for _,row in df.iterrows()]
            texts.insert(0, f'{year}年{company}（{stock_id}）月度股票行情数据。')
            fw.write(''.join(texts)  + '\n')

## 示例代码8.4 基于向量查询的文本搜索引擎

In [132]:
import numpy as np
from sentence_transformers import SentenceTransformer

class TextSearchEngine():
    def __init__(self, text_encoder_path="./fm/all-MiniLM-L6-v2"):
        self.encoder = SentenceTransformer(text_encoder_path)
        self.embeddings = np.array([]) 
        self.texts = []

    def _txt2vec(self, texts):
        return self.encoder.encode(texts)
        
    def index(self, texts): # 新增文本向量表示
        keys = [text.split('。')[0] for text in texts] # 将每条记录的第一句话向量化 
        new_embeddings = self._txt2vec(keys)
        if len(self.embeddings) == 0:
            self.embeddings = new_embeddings
        else:
            self.embeddings = np.vstack((self.embeddings, new_embeddings))
        self.texts.extend(texts)
        
    def search(self, query, top_k=2): # 查找相似文本
        query_embedding = self._txt2vec([query])
        # 此处模型已经保证embedding范数为1
        scores = np.dot(self.embeddings, query_embedding.T).flatten()
        top_indices = np.argsort(scores)[-top_k:][::-1]
        hits = [self.texts[i] for i in top_indices]
        return hits

## 示例代码8.5 将公司股价知识库向量化

In [133]:
stock_data_path = 'stock_data_tfshare.txt' 
texts = open(stock_data_path).readlines()
texts = [line.strip('\r\n') for line in texts]
searcher = TextSearchEngine()
searcher.index(texts)

## 示例代码8.6 相关文档检索

In [67]:
query = "分析2025年工商银行的股价走势和特点"
hits = searcher.search(query=query, top_k=2)
for text in hits:
    print(text)

2025年工商银行（601398）月度股票行情数据。日期2025-01-27，收盘价6.345元。日期2025-02-28，收盘价6.395元。日期2025-03-31，收盘价6.415元。日期2025-04-30，收盘价6.535元。日期2025-05-30，收盘价6.595元。日期2025-06-30，收盘价7.115元。日期2025-07-31，收盘价7.25元。日期2025-08-29，收盘价7.12元。日期2025-09-30，收盘价6.99元。日期2025-10-31，收盘价7.47元。日期2025-11-28，收盘价7.8元。日期2025-12-31，收盘价7.761元。
2024年工商银行（601398）月度股票行情数据。日期2024-01-31，收盘价4.245元。日期2024-02-29，收盘价4.405元。日期2024-03-29，收盘价4.355元。日期2024-04-30，收盘价4.505元。日期2024-05-31，收盘价4.505元。日期2024-06-28，收盘价4.775元。日期2024-07-31，收盘价5.232元。日期2024-08-30，收盘价5.362元。日期2024-09-30，收盘价5.562元。日期2024-10-31，收盘价5.422元。日期2024-11-29，收盘价5.532元。日期2024-12-31，收盘价6.302元。


## 示例代码8.7 以相关文档为上下文让大模型给出回答

In [68]:
llm = LocalLLMClient('local_llms/Qwen3-0.6B')

### 不提供上下文，让模型直接回答

In [70]:
prompt = f"{query}"
resp = llm.answer(prompt)
print(resp)

2025年是工商银行（Bank of China）发展历程中的一个关键节点，其股价走势和特点可以从以下几个方面进行分析：

---

### 一、2025年行业背景与政策影响

1. **宏观经济环境**  
   - 2025年全球经济面临不确定性，但中国仍处于稳增长阶段，政策支持（如“十四五”规划、降息降准等）可能为工行提供稳定增长空间。  
   - 工行作为国内领先的国有大型商业银行，其业务模式（如零售、信贷、支付等）在政策支持下仍具有优势。

2. **金融市场变化**  
   - 银行间竞争加剧，工行可能通过优化产品结构、提升服务效率来应对挑战。  
   - 数字金融、金融科技的发展可能推动工行在数字化转型上的表现。

---

### 二、2025年股价走势特点

1. **板块轮动与分化**  
   - 工行作为国有银行，可能在不同业务板块中表现分化：  
     - **零售与消费**：工行的消费金融、线上支付等业务可能受政策支持和市场需求推动。  
     - **信贷与投资**：工行的信贷业务可能在政策导向下保持稳定增长，但需关注利率、信用风险等因素。  
     - **国际化**：若工行国际化进程加快，其海外业务可能在2025年迎来一定调整。

2. **市场情绪与信心**  
   - 工行作为国有大型银行，其股价可能受到市场对经济复苏、政策支持的预期影响。  
   - 2025年可能面临市场波动，但若政策持续利好，工行的股价可能稳定或上升。

3. **技术因素**  
   - 工行可能在技术层面（如大数据、云计算、金融科技）的投入，可能推动其股价上涨。  
   - 2025年可能有更多数字化转型项目落地，从而带动工行股价。

---

### 三、2025年可能的股价趋势预测

- **短期（1-3个月）**：  
  - 工行可能因市场预期或政策支持而小幅上涨，尤其是在消费金融、数字化转型等板块表现较好。  
  - 需关注市场对利率变化、汇率波动等风险因素的影响。

- **中期（4-6个月）**：  
  - 工行可能在业务板块中实现结构性调整，例如在零售、消费、信贷等领域进一步优化产品结构。  
  - 若国际化进程加快，海外业务可能在2025年迎来一定调整，但整体仍具增长潜力。

- **长期（6-12个月

### 在提示中包含上下文

In [74]:
context = '\n'.join(hits)
prompt = f"请结合以下文本内容回答问题：{context}。\n问题为：{query}"
resp = llm.answer(prompt)
print(resp)

2025年工商银行（601398）的股价走势呈现出以下特点：

1. **震荡上涨**：2025年1月27日，收盘价为6.345元，随后在2月28日、3月31日、4月30日等日期上涨至6.535元、6.415元、6.595元等，显示出一定的上涨趋势。

2. **波动性**：股价在不同日期的波动幅度有所变化，例如1月27日收盘价为6.345元，随后在2月28日、3月31日等日期有所上涨，但整体波动性依然存在。

3. **趋势延续**：从2025年1月27日到2025年7月31日，股价经历了从6.345元到7.25元的上涨，随后在8月29日、9月30日、10月31日、11月28日和12月31日逐步回落至7.12元、6.99元、7.47元、7.8元和7.761元，整体走势较为平稳。

4. **市场表现**：2025年11月28日的收盘价为7.8元，表明在该月的市场中，工商银行股价表现较为稳定，可能反映了市场对该公司的信心增强。

综上所述，2025年工商银行的股价走势呈现出震荡上涨的趋势，整体波动性较低，市场表现较为稳健。


## 课堂练习-8-1
TextSearchEngine类中的index方法不检查文档是否已经在库中，从而有文档重复入库的风险。请修改该方法的实现，只对尚未入库的文本做向量化，已在库中的则跳过。

## 智能体应用案例 自动化股价获取及邮件通知

## 示例代码8.8 定义可供智能体使用的工具箱类AgentToolkit

In [135]:
class AgentToolkit():
    def __init__(self):
        self.tools = {}
        
    def add(self, tool_func, tool_desp): 
        """
        添加工具。
        Args:
            tool_func: 实现该工具的函数对象
            tool_desp: 关于该工具的描述
        Returns:
            None: 无返回值
        """
        self.tools[tool_func.__name__] = {"func": tool_func, "info": tool_desp}
    
    def get(self, tool_name): 
        """
        调用工具
        """
        return self.tools.get(tool_name, {}).get("func", None)

    def describe(self): 
        """
        获取所有工具的描述
        """
        desp = [f"{tool_name}: {tool['info']}" for tool_name,tool in self.tools.items()]
        return "\n".join(desp)


# 测试代码
def hello(name):
    return f'Hello {name}'
    
atk = AgentToolkit()
atk.add(max, 'compare two input numbers and returns the larger')
atk.add(hello, 'returns a hello message to a give name')
print(atk.describe())

max: compare two input numbers and returns the larger
hello: returns a hello message to a give name


## 示例代码8.9 定义一个获取昨日股价的函数get_yesterday_stock
因akshare调用不稳定，改为调用TFshare（见**示例代码8.3** 方案2）

In [77]:
from datetime import date, timedelta
#import akshare as ak  

def get_yesterday_stock(stock_id):
    target_date = date.today() - timedelta(days=1) # 以调用时刻反推昨日
    target_date = target_date.strftime("%Y%m%d")
    #df = ak.stock_zh_a_hist(symbol=stock_id, period='daily', start_date=target_date, end_date=target_date, adjust='qfq')
    df = TFShare.stock_zh_a_hist(symbol=stock_id, period='daily', start_date=target_date, end_date=target_date, adjust='qfq')
    record = df.loc[0].to_dict()
    record['日期'] = record['日期'].strftime("%Y%m%d")
    record = [f"{key}{value}" for key, value in record.items()]
    record = '，'.join(record) + '。'
    return record

# 函数调用示例
print(get_yesterday_stock('601398'))

日期20260902，开盘8.19，收盘8.22，最高8.29，最低8.14，成交量3717999.0。


## 示例代码8.10 定义一个发送邮件的函数send_email

In [79]:
import smtplib
from email.mime.text import MIMEText

def send_email(subject: str, content: str, receiver: str):
    sender = "xxxx@xxx.com"  # 发件箱
    authcode = "xxxx"  # 发件箱授权码（注意不是登陆密码），在邮箱SMTP设置页面查看
    smtp_host = "xxxx"  # 发件箱SMTP服务器地址（新浪为smtp.sina.com，163为smtp.163.com，qq为smtp.qq.com）
    msg = MIMEText(content, "plain", "utf-8")  # 创建邮件对象
    msg["Subject"], msg["From"], msg["To"] = subject, sender, receiver
    try:
        with smtplib.SMTP_SSL(smtp_host, 465) as server:
            server.login(sender, authcode)
            server.send_message(msg)
        return "邮件已发送"
    except Exception as e:
        return f"邮件发送失败: {e}"

# 函数调用示例
send_email(subject="test", content="hello world", receiver="xxx@126.com")

'邮件发送失败: [Errno 8] nodename nor servname provided, or not known'

## 示例代码8.11 为智能体配置工具箱

In [80]:
toolkit = AgentToolkit()
toolkit.add(get_yesterday_stock, "获取指定股票的昨日交易数据，参数stock_id表示股票代码")
toolkit.add(send_email, "发送电子邮件，输入邮件主题和内容，发送邮件到指定收件人。参数subject表示邮件主题，content表示邮件内容，receiver表示收件人邮箱地址")
print(toolkit.describe())

get_yesterday_stock: 获取指定股票的昨日交易数据，参数stock_id表示股票代码
send_email: 发送电子邮件，输入邮件主题和内容，发送邮件到指定收件人。参数subject表示邮件主题，content表示邮件内容，receiver表示收件人邮箱地址


## 示例代码8.12 让大模型扮演智能体“大脑” 的提示词

In [82]:
def apply_chat_template(prompt, toolkit_desp, previous_state=''):
    prompt_template = f"""
    你是一个智能助理，能够调用外部工具来完成相关的任务。
    可以使用的工具如下：
    {toolkit_desp}

    你的回答必须是以下两种形式之一，且不能同时进行：
    1、如果你需要调用工具来获取信息，按照“行动：[工具名称];[JSON格式，参数名:参数值]”的格式进行回答。
    2、如果你已经有足够的信息来回答用户的问题，按照“回答：[你的回答内容]”的格式进行回答。

    任务如下：
    {prompt}
    之前返回结果：
    {previous_state}
    """
    return prompt_template
# 函数调用示例
task_prompt = "查询工商银行（股票代码：601398）昨日的股票交易数据，并将数据发送到邮箱xxxx@ruc.edu.cn"
res = apply_chat_template(task_prompt, toolkit.describe())
res

'\n    你是一个智能助理，能够调用外部工具来完成相关的任务。\n    可以使用的工具如下：\n    get_yesterday_stock: 获取指定股票的昨日交易数据，参数stock_id表示股票代码\nsend_email: 发送电子邮件，输入邮件主题和内容，发送邮件到指定收件人。参数subject表示邮件主题，content表示邮件内容，receiver表示收件人邮箱地址\n\n    你的回答必须是以下两种形式之一，且不能同时进行：\n    1、如果你需要调用工具来获取信息，按照“行动：[工具名称];[JSON格式，参数名:参数值]”的格式进行回答。\n    2、如果你已经有足够的信息来回答用户的问题，按照“回答：[你的回答内容]”的格式进行回答。\n\n    任务如下：\n    查询工商银行（股票代码：601398）昨日的股票交易数据，并将数据发送到邮箱xxxx@ruc.edu.cn\n    之前返回结果：\n    \n    '

## 示例代码8.13 智能体主体架构

In [126]:
import json
from openai import OpenAI

class Agent:
    def __init__(self, llm, toolkit):
        self.toolkit = toolkit
        self.llm = llm
        self.max_steps = 5
        
    def _process(self, prompt):
        previous_results = []
        tk_desp = self.toolkit.describe()
        for i in range(self.max_steps):
            full_prompt = apply_chat_template(prompt, tk_desp, '\n'.join(previous_results))
            resp = self.llm.answer(full_prompt).strip() 
            print(f"Step {i+1}. llm> {resp}")
            if resp.startswith("回答："):
                return {'status':True, 'message': resp[len("回答："):]}
            elif resp.startswith("行动："): 
                action_part = resp[len("行动："):]
                tool_name, param_str = action_part.split(';', 1)
                param_str = json.loads(param_str)
                tool_func = self.toolkit.get(tool_name)
                if tool_func:
                    res = tool_func(**param_str)
                    previous_results.append(res) # 收集外部工具的执行结果
                else:
                    return {'status':False, 'message':f"未找到工具 {tool_name}"}
            else:
                return {'status':False, 'message': "无法解析的响应格式"}
        return {'status':False, 'message': "达到最大步骤数，未能得到最终回答。"}

    def answer(self, prompt):
        resp = self._process(prompt)
        return resp['message']

## 示例代码8.14 调用Agent查询股价并发送到指定邮箱

In [127]:
task_prompt = "查询工商银行（股票代码：601398）昨日的股票交易数据，并将数据发送到邮箱xxxx@ruc.edu.cn"
agent = Agent(llm, toolkit)
resp = agent.answer(task_prompt)
resp

Step 1. llm> 行动：get_yesterday_stock;{"stock_id": "601398"}
Step 2. llm> 行动：send_email;{"subject": "工商银行（601398）昨日股票交易数据", "content": "日期：20260902\n开盘价：8.19\n收盘价：8.22\n最高价：8.29\n最低价：8.14\n成交量：3717999.0", "receiver": "xxxx@ruc.edu.cn"}
Step 3. llm> 行动：send_email;{"subject": "工商银行（601398）昨日股票交易数据", "content": "日期：20260902\n开盘价：8.19\n收盘价：8.22\n最高价：8.29\n最低价：8.14\n成交量：3717999.0", "receiver": "xxxx@ruc.edu.cn"}
Step 4. llm> 行动：send_email;{"subject": "工商银行（601398）昨日股票交易数据", "content": "日期：20260902\n开盘价：8.19\n收盘价：8.22\n最高价：8.29\n最低价：8.14\n成交量：3717999.0", "receiver": "xxxx@ruc.edu.cn"}
Step 5. llm> 行动：send_email;{"subject": "工商银行（601398）昨日股票交易数据", "content": "日期：20260902\n开盘价：8.19\n收盘价：8.22\n最高价：8.29\n最低价：8.14\n成交量：3717999.0", "receiver": "xxxx@ruc.edu.cn"}


'达到最大步骤数，未能得到最终回答。'

## 示例代码8.16 尝试让Agent回答一般问题

In [136]:
task_prompt = "最小的鸟类是什么？"
resp = agent.answer(task_prompt)
print(resp)

Step 1. llm> 回答：最小的鸟类是蜂鸟，其中尤以蜜蜂蜂鸟（学名：Mellisuga helenae）为最小。这种蜂鸟主要分布于古巴，体长约5至6厘米，体重仅约1.6至2克，是世界上已知体型最小的鸟类。
最小的鸟类是蜂鸟，其中尤以蜜蜂蜂鸟（学名：Mellisuga helenae）为最小。这种蜂鸟主要分布于古巴，体长约5至6厘米，体重仅约1.6至2克，是世界上已知体型最小的鸟类。


## 示例代码8.17 尝试让Agent订机票

In [138]:
task_prompt = "我明天从北京去深圳出差，请订一张早上10点出发的直飞机票。"
resp = agent.answer(task_prompt)
print(resp)

Step 1. llm> 回答：我无法为您预订机票，因为当前没有提供订票工具。建议您通过航空公司官网或旅行平台自行预订。
我无法为您预订机票，因为当前没有提供订票工具。建议您通过航空公司官网或旅行平台自行预订。


## 课堂练习-8-2
在当前代码中，工具函数的执行（第25行）没有任何保护。一旦工具内部抛出异常，整个智能体会立即终止，用户既得不到回答，也看不到已经完成的中间步骤。请为智能体增加执行阶段的容错能力。

## 课堂练习8-3

案例中的智能体无法订票。现定义一个book_flight函数，描述为预订飞机票工具，但实际仅返回“订票已完成”。将该工具加入智能体工具箱之后，智能体会如何响应？

```python
def book_flight(departure_city, arrival_city, departure_time, flight_type):
    return "订票已完成"
    
toolkit.add(book_flight, "预订飞机票。参数departure_city表示出发地，arrival_city表示目的地, departure_time表示出发时间, flight_type表示航班类型")

```